In [187]:
import numpy as np
import pandas as pd

In [188]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [189]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [190]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [191]:
movies = movies.merge(credits,on='title')

In [192]:
movies.shape

(4809, 23)

In [193]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [194]:
# *******movies['original_language'].value_counts() gives the count of languages present in dataset ******* in here we don't select it bcz it has 4500 movies in englsish so more than 90% of the movies are in english that is not going to affect much so we will ignore language col
# movies.info() ---> gives entire details about row and cols and etc

In [195]:
# selecting important columns that affects more to recomend a movie form 23 columns
# genres
# id or movie_id both are same
# keywords
# title
# overview
# cast
# crew
movies = movies[["movie_id","title","overview","genres","keywords","cast","crew"]]

In [196]:
# I'm trying to remove missing data and duplicate data
movies.isnull().sum()
# i see there are three missing values in overview so i will remove those rows

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [197]:
movies.dropna(inplace=True)
# inplace = True --> it modifies movies data frame directly instead returning new copy

In [198]:
movies.shape

(4806, 7)

In [199]:
movies.duplicated().sum()

# there are no rows which have duplicate values

np.int64(0)

In [200]:
# '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'
# from above to below i need to convert
# ["Action","Adventure","Fantasy","Science Fiction"]
# so i am going to create a helper function that converts them in to something like list of strings
# notice that it is a string of list for that i am importing a library "ast" which is in python

In [201]:
import ast
# ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]') --> convertes to a list of dictionories
def converter(lst):
    lst_of_genre = []
    for i in ast.literal_eval(lst):
        lst_of_genre.append(i["name"])
    return lst_of_genre

In [202]:
movies["genres"] = movies["genres"].apply(converter)

In [203]:
# do the same for keywords also to extract keywords used
# movies["keywords"][0]
movies["keywords"] = movies["keywords"].apply(converter)

In [204]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [205]:
# do the same here to extract cast names
# movies["cast"][0] --> gives casts of first movie but i want only top three casts from this cast so i will selct first three casts
# and how do i do that? i will select first three dictionaries in that list and i will extract name from it.
def converter3(lst):
    lst_of_genre = []
    counter = 0
    for i in ast.literal_eval(lst):
        if counter != 3:
            lst_of_genre.append(i["name"])
            counter += 1
        else:
            break
    return lst_of_genre


In [206]:
movies["cast"] = movies["cast"].apply(converter3)

In [207]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [208]:
# now we have crew we have to extract the important info only

In [209]:
# movies['crew'][0]
# i am only intersted to get the name of director so i am only interested who's job is Dirctor and i will only concentrate on that job
# below is the function 
def find_director(str):
    director_list = []
    for item in ast.literal_eval(str):
        if(item["job"].lower() == "director"):
            director_list.append(item["name"])
            break; # only one director so once director is found no need to proceed for another dictionaries
    return director_list

In [210]:
movies["crew"] = movies["crew"].apply(find_director)

In [211]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [212]:
 # now the overview is a string datatype so i also want them to be stored in list with separted by space
# we are going to split it it will be list of words with words split based on space

movies["overview"] = movies["overview"].apply(lambda x: x.split())

In [213]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [115]:
# now i want to create a tag
# that tag is like a paragraph of all these columns --> (overview,genres,keywords,cast,crew)
# Example intuition
# Avatar tag:
# space war future adventure sam worthington james cameron
# Another  movie:
# space alien future adventure hero sam neill

# 👉 They share words like:

# space
# future
# adventure

# So the system says:

# “These movies are similar”
# Why not keep separate columns?

# Because ML models like:

# CountVectorizer
# TF-IDF
# cosine similarity

# work best on single text input, not multiple columns.

# So we combine everything into one “text field”.

In [214]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
# ex:- Science Fiction ----> ScienceFiction

In [117]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [215]:
movies["tags"] = movies["overview"] + movies["genres"] + movies["keywords"] + movies["cast"] + movies ["crew"]

In [119]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [216]:
# now i don't need these columns ---> (title	overview	genres	keywords	cast	crew)
new_df = movies[["movie_id","title","tags"]]
new_df

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4804,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4805,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4806,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4807,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


In [217]:
# now let's convert the tags column data type into a string dtype
new_df["tags"] = new_df["tags"].apply(lambda x:" ".join(x))

In [218]:
new_df.tags[0]
# this paragraph is in this format
# overview	genres	keywords	cast(specifically ---> top casts like hero heroin)	crew(director only)
# let's convert all paragraphs in this column(tags column) into lower case text (recomended)

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [123]:
new_df["tags"] = new_df["tags"].apply(lambda x: x.lower())

In [219]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [220]:
# we are done with data pre-processing
# now we are going to proceed with vectorization
new_df["tags"][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [221]:
from sklearn.feature_extraction.text import CountVectorizer

In [222]:
# CountVectorizer from scikit-learn converts text into numbers so machine learning models can understand it.
# fit() learns all unique words (vocabulary) from the dataset.
# transform() converts each sentence into a vector of word counts.
# Every sentence uses the same word order based on the learned vocabulary.
# Example vocabulary:
    # [action, adventure, hero, romance]
# Sentence: "action adventure hero"
# becomes:

# [1, 1, 1, 0]
# meaning:

# action appears once → 1
# adventure appears once → 1
# hero appears once → 1
# romance does not appear → 0
# 0 means the word is absent in that sentence.
# .toarray() converts the result into a normal NumPy array.
# This whole process is called the Bag of Words technique.


# step:1
cv = CountVectorizer(max_features=5000,stop_words="english");
vectors = cv.fit_transform(new_df["tags"]).toarray()

In [223]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [129]:
list(cv.get_feature_names_out())

['000',
 '007',
 '10',
 '100',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '18th',
 '19',
 '1930s',
 '1940s',
 '1950',
 '1950s',
 '1960s',
 '1970s',
 '1980',
 '1980s',
 '1985',
 '1990s',
 '1999',
 '19th',
 '19thcentury',
 '20',
 '200',
 '2009',
 '20th',
 '24',
 '25',
 '30',
 '300',
 '3d',
 '40',
 '50',
 '500',
 '60',
 '60s',
 '70',
 '70s',
 'aaron',
 'aaroneckhart',
 'abandoned',
 'abducted',
 'abigailbreslin',
 'abilities',
 'ability',
 'able',
 'aboard',
 'abuse',
 'abusive',
 'academy',
 'accept',
 'accepted',
 'accepts',
 'access',
 'accident',
 'accidental',
 'accidentally',
 'accompanied',
 'accomplish',
 'account',
 'accountant',
 'accused',
 'ace',
 'achieve',
 'act',
 'acting',
 'action',
 'actionhero',
 'actions',
 'activist',
 'activities',
 'activity',
 'actor',
 'actors',
 'actress',
 'acts',
 'actual',
 'actually',
 'adam',
 'adams',
 'adamsandler',
 'adamshankman',
 'adaptation',
 'adapted',
 'addict',
 'addicted',
 'addiction',
 'adolescence',
 'adolescent'

In [130]:
len(cv.get_feature_names_out())

5000

In [131]:
# if you observe in the features there are features that have the same meaning like
# actor ---> actors 
# adam ---> adams
# addict ---> addicted and so on .....
# to avoid this i need to perform stemming

In [132]:
# ["loved","loving","love"]
# "love" stemmed word
# to do so we need a library Natural Language toolkit library(NLTK)
# pip install nl

In [133]:
import nltk

In [134]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [135]:
ps.stem("loving")

'love'

In [136]:
def stem(text):
    lst = []
    for item in text.split():
        lst.append(ps.stem(item))
    return " ".join(lst)

In [137]:
print("in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron")
# before stemmig

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron


In [138]:
stem("in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron")
# after stemming

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [139]:
# applying stemming to each tag(movie tag) and then saving back in the data frame
new_df["tags"] = new_df["tags"].apply(stem)

In [140]:
# repeat the step1 mentioned above(creating a fearure vocabulary from tags column with variable name cv)
vectors = cv.fit_transform(new_df["tags"]).toarray()

In [141]:
list(cv.get_feature_names_out())

['000',
 '007',
 '10',
 '100',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '17th',
 '18',
 '18th',
 '18thcenturi',
 '19',
 '1910',
 '1920',
 '1930',
 '1940',
 '1944',
 '1950',
 '1950s',
 '1960',
 '1960s',
 '1970',
 '1970s',
 '1971',
 '1974',
 '1976',
 '1980',
 '1985',
 '1990',
 '1999',
 '19th',
 '19thcenturi',
 '20',
 '200',
 '2003',
 '2009',
 '20th',
 '21st',
 '23',
 '24',
 '25',
 '30',
 '300',
 '3d',
 '40',
 '50',
 '500',
 '60',
 '70',
 '80',
 'aaron',
 'aaroneckhart',
 'abandon',
 'abduct',
 'abigailbreslin',
 'abil',
 'abl',
 'aboard',
 'abov',
 'abus',
 'academ',
 'academi',
 'accept',
 'access',
 'accid',
 'accident',
 'acclaim',
 'accompani',
 'accomplish',
 'account',
 'accus',
 'ace',
 'achiev',
 'acquaint',
 'act',
 'action',
 'actionhero',
 'activ',
 'activist',
 'activities',
 'actor',
 'actress',
 'actual',
 'ad',
 'adam',
 'adamsandl',
 'adamshankman',
 'adapt',
 'add',
 'addict',
 'adjust',
 'admir',
 'admit',
 'adolesc',
 'adopt',
 'ador',
 'adrienbrodi',
 'adult'

In [142]:
print(new_df["tags"].count())
# here we have total 4806 movies so those many vectors
print(len(vectors))

4806
4806


In [143]:
# now to find the related movie i have to find the distance from one movie to another for all
# more the distance lesser the similarity
# here i am going to calculate cosine similarity rather than eucledean distance 
# when data is more or for higher dimension eucledean distance is not a good measure because 
# in eucledean distance you calculate distance between each vector tip.
# but in cosine similarity you find angle b/w them that gives more accurate value

In [144]:
from sklearn.metrics.pairwise import cosine_similarity

In [145]:
similarity = cosine_similarity(vectors)
print(similarity.shape)

(4806, 4806)


In [146]:
# (4806, 4806)
# so here we have to calculate distance from each vector with all 4805 vectors including it self
# it gives 4806
# like 1st movie with all other movie distance (1,4806) again 2nd movie with (2,4806) so on... till (4806,4806)

In [147]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ], shape=(4806,))

In [148]:
similarity[1]

array([0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
       0.02615329], shape=(4806,))

In [149]:
similarity[2]

array([0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
       0.        ], shape=(4806,))

In [150]:
# here in 0th and 1st elemement observe 0th index and 1st index are 1 respectively in 0th and
# 1st index element i'e cosine similarity of element with itself is going to be same so 
# cos(0) = 1
# so diagonal will always be one in our case

In [151]:
# now when some one search for any movie like say avatar 1st movie i will sort the array in
# descending order. when it is in ascending order first 5 will be the nearest to that movie

# new_df["title"] == "Avatar" #---> this gives the details of data frame with true or false based on title has movie name "Avatar"
# new_df[new_df["title"] == "Avatar"]#---> this gives the row where movie avatar is present
# new_df[new_df["title"] == "Avatar"].index #---> this gives the index object with index vlaue where avatar movie present and dtype of it
# new_df[new_df["title"] == "Spectre"].index[0] ---> index has only one value bcz all movies are unique so it will have one value that value is located in 0th index

In [152]:
# sort those distance array
# as discused before if we sort in descending top 5 will be the similar movies
# but how we are going to capture it's index because that index helps us in fetching that movie
# from data frame so we have to capture their index also while we sort them so like
# array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
#        0.        ], shape=(4806,))
# 0th movie has 100% similarity with it self 1st movie has 0.083 similarity with movie 1
# so it's index is associated with it's movie so we shouldn't loose it
# we are going to something like ----below----:
# sort(similarity[0]) ---> sorts all but looses it's index
# enumerate(similarity[0])  ---> creates a object of similarities 
# list(enumerate(similarity[0])) ---> gives touples of each value with index captured in it
# key=lambda x: x[1] --> sort based on 2nd(1st indexed) value in the touple
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x: x[1])[1:6]
# will use above line in the function

[(1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

In [153]:
def recomended(movie):
    movie_index = new_df[new_df["title"] == movie].index[0]
    distances = similarity[movie_index]
    # sort those distance array
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for movie in movies_list:
        # print(movie[0])
        print(new_df.iloc[movie[0]].title)

In [154]:
# new_df.iloc[1214].title

In [155]:
recomended("Batman Begins")

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [156]:
# now our task is to convert that into an website
import pickle

In [157]:
pickle.dump(new_df.to_dict(),open("movies_dict.pkl","wb"))

In [158]:
pickle.dump(similarity,open("similarity.pkl","wb"))